In [1]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import random
from itertools import product
import optuna
import numpy as np
from scipy.stats import norm

from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search, estimate_single_config


In [2]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [4]:
def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30) -> pd.DataFrame:
    """Return AR(1) innovations (residuals) for each column of X."""
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()
        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()
        X_innov.loc[tmp.index, col] = res.resid

    return X_innov

In [3]:
# set seed
random.seed(42)

# select a response variable (either marekt return or sp500 return)
y = response_variables['vwretx']  # or 'sprtrn' for SP500 returns or vwretx

# # transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# User-selected number of each 
num_topics = len(topic_cols)  
num_stocks = 0

# Randomly sample (without replacement), limited by available count
selected_topics = random.sample(topic_cols, min(num_topics, len(topic_cols)))
selected_stocks = random.sample(stock_cols, min(num_stocks, len(stock_cols)))

# Final filtered dataframe
X = X[selected_topics + selected_stocks]

# Convert stock returns to log returns: log(1+r)
X[selected_stocks] = np.log(X[selected_stocks] + 1)

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [4]:
from itertools import product
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

def grid_search(
    X,
    y,
    param_grid,
    verbose=True,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    return_details=True,
):
    """
    Parallel grid search.

    Parameters
    ----------
    return_details : bool, default True
        If False, skip collecting and concatenating window-level details
        (MUCH faster and lower memory usage).

    Returns
    -------
    results_df : pd.DataFrame
        Summary metrics for each configuration.
    coefficients_df : pd.DataFrame or None
        Detailed coefficients if return_details=True, else None.
    """

    combos = list(product(
        param_grid["window_sizes"],
        param_grid["n_lags"],
        param_grid["lambdas"],
    ))

    if verbose:
        print(f"Testing {len(combos)} configurations...")

    # -------------------------------------------------
    # Wrapper to prevent single failure from killing run
    # -------------------------------------------------
    def _safe_run(args):
        w, L, lam = args
        try:
            return estimate_single_config(X, y, w, L, lam)
        except Exception as e:
            if verbose:
                print(f"❌ Failed config (w={w}, L={L}, λ={lam}): {e}")
            return None

    iterator = tqdm(combos, desc="Grid search") if verbose else combos

    # -------------------------------------------------
    # Parallel execution
    # -------------------------------------------------
    results = Parallel(n_jobs=n_jobs, backend=backend, prefer=prefer)(
        delayed(_safe_run)(args) for args in iterator
    )

    # -------------------------------------------------
    # Aggregate results
    # -------------------------------------------------
    summary_list = []
    details_list = [] if return_details else None

    for res in results:
        if res is None:
            continue

        summary_list.append(res.get("summary", {}))

        if return_details:
            det = res.get("details", None)
            if det is not None and not det.empty:
                details_list.append(det)

    # -------------------------------------------------
    # Build summary DataFrame
    # -------------------------------------------------
    results_df = pd.DataFrame(summary_list)

    if not results_df.empty and "r2_oos_stage2" in results_df.columns:
        results_df = results_df.sort_values(
            "r2_oos_stage2", ascending=False
        ).reset_index(drop=True)

    # -------------------------------------------------
    # Build details DataFrame (optional)
    # -------------------------------------------------
    coefficients_df = None
    if return_details and details_list:
        coefficients_df = pd.concat(details_list, ignore_index=True)

    # -------------------------------------------------
    # Reporting
    # -------------------------------------------------
    if verbose:
        print("\n" + "=" * 80)
        print("GRID SEARCH COMPLETE")
        print("=" * 80)

        if "kappa" in results_df.columns:
            n_failed = results_df["kappa"].isna().sum()
            if n_failed > 0:
                print(f"⚠️  {n_failed}/{len(results_df)} configurations failed")

    return results_df, coefficients_df


## Grid Search

In [5]:
# Objective Function
def objective_function(row):
    weight = 2 * (norm.cdf(abs(row['kappa_tstat'])) - 0.5)
    return row['r2_insample_stage2'] * weight
# -----------------------------
# Stopping criteria (NEW)
# -----------------------------
max_iterations = 20          # maximum refinement iterations
improvement_tol = 1e-6       # stop if best objective improves by less than this

# -------------------------------------------------
# Objective function: maximize R^2 only
# -------------------------------------------------
def objective_function(row, r2_col="r2_insample_stage2"):
    r2 = pd.to_numeric(row.get(r2_col, np.nan), errors="coerce")
    if not np.isfinite(r2):
        return -1.0
    return r2

# -------------------------------------------------
# Single grid search
# -------------------------------------------------
param_grid = {
    "window_sizes": [36, 52, 78, 104, 150, 200],
    "n_lags": [1], #, 4, 8, 12],
    "lambdas": [0.0001, 0.00001],
}

# Run grid search once
summary_df, details_df = grid_search(X, y, param_grid, verbose=True)

# Compute objective
summary_df["objective"] = summary_df.apply(objective_function, axis=1)

# -------------------------------------------------
# Select best configuration
# -------------------------------------------------
valid = summary_df[summary_df["objective"] > -1.0]

if valid.empty:
    best_overall = None
    print("No valid configurations found.")
else:
    best_overall = valid.loc[valid["objective"].idxmax()]
    print(best_overall[["window_size", "n_lags", "lambda", "objective"]])

# -------------------------------------------------
# Return results
# -------------------------------------------------
summary_df, details_df, best_overall


Testing 12 configurations...


Grid search: 100%|██████████| 12/12 [00:00<00:00, 15.78it/s]



GRID SEARCH COMPLETE
window_size    78.000000
n_lags          1.000000
lambda          0.000100
objective       0.000042
Name: 8, dtype: float64


/var/folders/hb/bc0srnt52zx6lbw4t2k_t_k80000gn/T/ipykernel_97527/2362108959.py:95: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  coefficients_df = pd.concat(details_list, ignore_index=True)


(    window_size  n_lags   lambda  r2_insample_stage1  r2_oos_stage1  \
 0           200       1  0.00010            0.761294      -1.384968   
 1           200       1  0.00001            0.890863      -5.522532   
 2           104       1  0.00010            0.955114      -1.341798   
 3           104       1  0.00001            0.994849      -1.916187   
 4            36       1  0.00001            0.980294      -0.817831   
 5            36       1  0.00010            0.976032      -0.804154   
 6           150       1  0.00010            0.883995      -1.745397   
 7           150       1  0.00001            0.992323      -4.854448   
 8            78       1  0.00010            0.973247      -1.044094   
 9            78       1  0.00001            0.993031      -1.279566   
 10           52       1  0.00001            0.987825      -0.886561   
 11           52       1  0.00010            0.979548      -0.819022   
 
     r2_insample_stage2  r2_oos_stage2         kappa   kappa_t

In [6]:
best_overall

window_size                   78.000000
n_lags                         1.000000
lambda                         0.000100
r2_insample_stage1             0.973247
r2_oos_stage1                 -1.044094
r2_insample_stage2             0.000042
r2_oos_stage2                 -0.004391
kappa                          0.004641
kappa_tstat                    0.274840
intercept                      0.000331
intercept_tstat                1.448967
n_observations              1806.000000
n_windows                   1808.000000
n_oos_predictions_stage2            NaN
objective                      0.000042
Name: 8, dtype: float64

In [ ]:
# import numpy as np
# import pandas as pd

results_all_rounds = []
# Initial grid
initial_param_grid = {
    
    # Evaluate objective
    summary_df['objective'] = summary_df.apply(objective_function, axis=1)

    # Track results
    results_all_rounds.append(summary_df.copy())

    # Pick best point of this iteration
    best_idx = summary_df['objective'].idxmax()
    best_row = summary_df.loc[best_idx]
    best_objective = float(best_row['objective'])

    print("Best this iteration:")
    print(best_row[['window_size', 'n_lags', 'lambda', 'objective']])

    # Early stopping checks
    if not np.isfinite(best_objective):
        print("Stopping: best objective is not finite (all candidates failed constraints).")
        break

    if i > 0:
        improvement = best_objective - prev_best_objective
        print(f"Improvement vs previous best: {improvement:.6g}")

        if improvement < improvement_tol:
            print(f"Stopping: improvement {improvement:.6g} < tolerance {improvement_tol:.6g}.")
            break

    prev_best_objective = best_objective

    # ----------------------------------------
    # Build new grid around best point
    # ----------------------------------------
    best_window = int(best_row['window_size'])
    best_n_lags = int(best_row['n_lags'])
    best_lambda = float(best_row['lambda'])

    # shrink window search range
    window_sizes_refined = list(range(best_window - 25, best_window + 26, 5))
    window_sizes_refined = [w for w in window_sizes_refined if w > 20]

    # shrink n_lags search range
    n_lags_refined = list(range(best_n_lags - 1, best_n_lags + 2))
    n_lags_refined = [l for l in n_lags_refined if l >= 1]

    # shrink lambda range 
    lambda_values_refined = np.linspace(
        0.8 * best_lambda,
        1.2 * best_lambda,
        num=5
    )

    # update grid for next iteration
    current_param_grid = {
        'window_sizes': window_sizes_refined,
        'n_lags': n_lags_refined,
        'lambdas': lambda_values_refined
    }

# ----------------------------------------
# Final evaluation
final_df = pd.concat(results_all_rounds, ignore_index=True)
final_df['objective'] = final_df.apply(objective_function, axis=1)
best_overall = final_df.loc[final_df['objective'].idxmax()]

print("\nBest hyperparameters after refinement:")
print(best_overall[['window_size', 'n_lags', 'lambda', 'objective']])

Best this iteration:
window_size    200.000000
n_lags          12.000000
lambda           0.000100
objective        0.000441
Name: 0, dtype: float64


SyntaxError: 'break' outside loop (4087755605.py, line 15)

In [10]:
# print for the 5 rows with the highest objective values r2 insample_stage2 and stage1, kappa ,kappa_tstat, lambda, window_size, n_lags
top_10 = final_df.nlargest(30, 'objective')
print(top_10[['r2_insample_stage2', 'r2_insample_stage1', 'r2_oos_stage2', 'kappa', 'kappa_tstat', 'lambda', 'window_size', 'n_lags']])

NameError: name 'final_df' is not defined

## Bayesian Optimization with optuna

For each hyperparameter triplet we do the following:

    - run stage1 and stage2
    - collect r2 in-sample stage2 and kappa
    - compute objective function: r2*kappa
    - let optuna try 150 random/learned samples to maximize the objective function

In [13]:
import optuna
import numpy as np
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# -----------------------------
# Focus region for economic meaning
# -----------------------------
TSTAT_COL = "kappa_tstat"
TSTAT_MIN = 1.96
TSTAT_MAX = 100.0

# Scoring / penalties (tune if needed)
SOFT_PENALTY_OUTSIDE = 2.0   # larger -> harder push into band
HARD_PRUNE_TOO_LARGE = True  # prune t-stat > TSTAT_MAX (often indicates instability)
EPS = 1e-12


def objective(trial):
    # Search space (keep yours; adjust if you like)
    window_size = trial.suggest_int("window_size", 30, 300, step=10)
    n_lags = trial.suggest_int("n_lags", 1, 10)
    lam = trial.suggest_float("lambda", 1e-6, 1e-3, log=True)

    # Run estimation
    res = estimate_single_config(
        X, y, window_size, n_lags, lam, 
        standardize=True, verbose=False, return_details=False
    )
    summary = (res or {}).get("summary", {})

    r2 = float(summary.get("r2_insample_stage2", np.nan))
    kappa = float(summary.get("kappa", np.nan))
    t = float(summary.get(TSTAT_COL, np.nan))

    # Store diagnostics
    trial.set_user_attr("r2_raw", r2)
    trial.set_user_attr("kappa_raw", kappa)
    trial.set_user_attr("kappa_tstat", t)

    # Basic validity
    if not (np.isfinite(r2) and np.isfinite(kappa) and np.isfinite(t)):
        raise optuna.TrialPruned()
    if r2 < 0 or kappa <= 0:
        # guide away without blowing up the optimizer
        return -0.1 - 0.01 * abs(r2) - 0.01 * abs(kappa)

    base = float(np.sqrt(max(r2, 0.0) * max(kappa, EPS)))
    trial.set_user_attr("base_score", base)

    # -----------------------------
    # Enforce / encourage the economically meaningful region:
    #   1.96 < t < 100
    # -----------------------------

    # If t-stat is *absurdly* large, treat as pathological and prune.
    if HARD_PRUNE_TOO_LARGE and t > TSTAT_MAX:
        trial.set_user_attr("t_band_status", "pruned_too_large")
        raise optuna.TrialPruned()

    # If not significant, keep trial but strongly penalize (so TPE learns)
    if t < TSTAT_MIN:
        # distance below the threshold, scaled to be comparable across magnitudes
        dist = (TSTAT_MIN - t) / TSTAT_MIN  # in (0, +inf)
        penalty = SOFT_PENALTY_OUTSIDE * dist
        score = base / (1.0 + penalty)
        trial.set_user_attr("t_band_status", "below_min_soft_penalty")
        trial.set_user_attr("penalty", penalty)
        return float(score)

    # In band -> best region, no penalty
    trial.set_user_attr("t_band_status", "in_band")
    trial.set_user_attr("penalty", 0.0)
    return base


# -----------------------------
# Sampler + pruner
# -----------------------------
sampler = TPESampler(
    seed=42,
    n_startup_trials=30,
    multivariate=True,
    constant_liar=True
)

pruner = MedianPruner(n_startup_trials=20, n_warmup_steps=5)

study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)


def print_best_callback(study, trial):
    if trial.number % 10 == 0 and study.best_trial is not None:
        bt = study.best_trial
        print(
            f"Trial {trial.number}: best={study.best_value:.4f} "
            f"(t={bt.user_attrs.get('kappa_tstat', np.nan):.2f}, "
            f"R2={bt.user_attrs.get('r2_raw', np.nan):.4f}, "
            f"kappa={bt.user_attrs.get('kappa_raw', np.nan):.4f})"
        )


study.optimize(
    objective,
    n_trials=250,
    n_jobs=6,
    callbacks=[print_best_callback],
    show_progress_bar=True
)

print("\n" + "=" * 60)
print("OPTIMIZATION RESULTS")
print("=" * 60)
print(f"Best Score (objective): {study.best_value:.6f}")
print(f"Best Params: {study.best_params}")

best_trial = study.best_trial
print("\nBest Trial Metrics:")
print(f"  R²:      {best_trial.user_attrs.get('r2_raw', np.nan):.6f}")
print(f"  Kappa:   {best_trial.user_attrs.get('kappa_raw', np.nan):.6f}")
print(f"  t-stat:  {best_trial.user_attrs.get('kappa_tstat', np.nan):.6f}")
print(f"  Base:    {best_trial.user_attrs.get('base_score', np.nan):.6f}")
print(f"  Status:  {best_trial.user_attrs.get('t_band_status', '')}")

# -----------------------------
# Reporting: "best within band" explicitly
# -----------------------------
in_band = [
    t for t in study.trials
    if t.value is not None
    and t.state == optuna.trial.TrialState.COMPLETE
    and (t.user_attrs.get("kappa_tstat", -np.inf) >= TSTAT_MIN)
    and (t.user_attrs.get("kappa_tstat", np.inf) <= TSTAT_MAX)
]

in_band.sort(key=lambda tr: tr.user_attrs.get("base_score", -np.inf), reverse=True)

print("\n" + "=" * 60)
print(f"TOP 10 TRIALS IN ECONOMIC BAND ({TSTAT_MIN}–{TSTAT_MAX})")
print("=" * 60)
if in_band:
    for i, tr in enumerate(in_band[:10], 1):
        print(
            f"{i}. base={tr.user_attrs.get('base_score', np.nan):.6f} | "
            f"obj={tr.value:.6f} | "
            f"R²={tr.user_attrs.get('r2_raw', np.nan):.4f} | "
            f"kappa={tr.user_attrs.get('kappa_raw', np.nan):.4f} | "
            f"t={tr.user_attrs.get('kappa_tstat', np.nan):.2f} | "
            f"params={tr.params}"
        )
else:
    print("No completed trials ended up in the target t-stat band.")

# -----------------------------
# Diagnostics: pruned counts & where mass is
# -----------------------------
states = [tr.state for tr in study.trials]
n_pruned = sum(s == optuna.trial.TrialState.PRUNED for s in states)
n_complete = sum(s == optuna.trial.TrialState.COMPLETE for s in states)

below = sum(
    (tr.state == optuna.trial.TrialState.COMPLETE)
    and (tr.user_attrs.get("kappa_tstat", -np.inf) < TSTAT_MIN)
    for tr in study.trials
)
above_pruned = sum(
    (tr.state == optuna.trial.TrialState.PRUNED)
    and (tr.user_attrs.get("t_band_status", "") == "pruned_too_large")
    for tr in study.trials
)

print("\n" + "=" * 60)
print("DIAGNOSTICS")
print("=" * 60)
print(f"Complete trials: {n_complete}/{len(study.trials)}")
print(f"Pruned trials:   {n_pruned}/{len(study.trials)}")
print(f"Complete but t < {TSTAT_MIN}: {below}")
print(f"Pruned for t > {TSTAT_MAX}:  {above_pruned}")


/var/folders/hb/bc0srnt52zx6lbw4t2k_t_k80000gn/T/ipykernel_94012/1481404408.py:81: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = TPESampler(
/var/folders/hb/bc0srnt52zx6lbw4t2k_t_k80000gn/T/ipykernel_94012/1481404408.py:81: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  sampler = TPESampler(
[I 2026-01-27 21:56:39,587] A new study created in memory with name: no-name-d0779fa7-6120-4b51-9de0-d43bdd982852


  0%|          | 0/250 [00:00<?, ?it/s]

Grid search:  33%|███▎      | 16/48 [01:02<00:19,  1.68it/s]

[I 2026-01-27 21:56:57,907] Trial 0 finished with value: -0.10000000705184019 and parameters: {'window_size': 80, 'n_lags': 7, 'lambda': 0.0005848406545712289}. Best is trial 0 with value: -0.10000000705184019.
Trial 0: best=-0.1000 (t=0.00, R2=-0.0000, kappa=0.0000)


Grid search:  33%|███▎      | 16/48 [01:06<00:19,  1.68it/s]

...

[I 2026-01-27 22:27:46,822] Trial 246 finished with value: 0.000587816295900945 and parameters: {'window_size': 220, 'n_lags': 4, 'lambda': 0.0002899189242668079}. Best is trial 209 with value: 0.01347082232216716.


Grid search:  33%|███▎      | 16/48 [32:03<00:19,  1.68it/s]

[I 2026-01-27 22:27:58,055] Trial 249 finished with value: -0.10000000000174565 and parameters: {'window_size': 170, 'n_lags': 1, 'lambda': 3.131883774938777e-05}. Best is trial 209 with value: 0.01347082232216716.


Grid search:  33%|███▎      | 16/48 [33:01<00:19,  1.68it/s]

[I 2026-01-27 22:28:56,939] Trial 243 finished with value: -0.10000000002522483 and parameters: {'window_size': 170, 'n_lags': 2, 'lambda': 1.115276018990673e-06}. Best is trial 209 with value: 0.01347082232216716.

OPTIMIZATION RESULTS
Best Score (objective): 0.013471
Best Params: {'window_size': 220, 'n_lags': 5, 'lambda': 0.0009820873597563983}

Best Trial Metrics:
  R²:      0.001967
  Kappa:   0.092248
  t-stat:  1.990179
  Base:    0.013471
  Status:  in_band

TOP 10 TRIALS IN ECONOMIC BAND (1.96–100.0)
1. base=0.013471 | obj=0.013471 | R²=0.0020 | kappa=0.0922 | t=1.99 | params={'window_size': 220, 'n_lags': 5, 'lambda': 0.0009820873597563983}
2. base=0.013201 | obj=0.013201 | R²=0.0019 | kappa=0.0904 | t=1.97 | params={'window_size': 220, 'n_lags': 5, 'lambda': 0.0009710256108686681}

DIAGNOSTICS
Complete trials: 250/250
Pruned trials:   0/250
Complete but t < 1.96: 248
Pruned for t > 100.0:  0


In [ ]:
res  = estimate_single_config(
    X, y,
    window_size= 10,
    n_lags=5,
    lambda_val=0.0025995,
    standardize=True,   
    verbose=True,
    return_details=True
)

In [ ]:
res